# Teszt

Az egyik állomásnál (VS) nagyon sok adat hiányzik. (NO2, SO2, PM10). Viszont ott több a rendelkezésre álló PM2.5 érték. A tesztben összehasonlítást végzünk, hogy a VS állomással vagy nélküle lesz jobb a modellek teljesítménye.


## Importok

A szükséges könyvtárak betöltése:

1. Használt modellek: RandomForestRegressor, HistGradientBoostingRegressor
2. pandas, numpy: adatkezelés
3. joblib: adatmentés, betöltés
4. sklearn.metrics: metrikákhoz

In [1]:
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

## Adat és artifactok betöltése

In [2]:
df = pd.read_parquet("../data/preprocessed/preprocessed_with_FE.parquet")
FEATURES = joblib.load("../artifacts/features.pkl")
LOCATION_MAPPING = joblib.load("../artifacts/location_mapping.pkl")

## Target létrehozás

In [3]:
df["pm25_next"] = (
    df.groupby("location")["pm25"].shift(-1)
)

TARGET = "pm25_next"

## NAN-ok törlése

In [4]:
columns = [
    "pm25_next",
    "pm25_lag1",
    "pm25_lag3",
    "pm25_lag6",
    "pm25_lag24",
    "pm25_roll6",
    "pm25_roll24",
    "pm25_trend_3h",
    "pm25_std_12h",
    "temp_change_3h",
    "humidity_change_3h",
    "wind_change_3h",
    "stagnation_hours_6h"
]

df = df.dropna(subset=columns)

## Adatok ellenőrzése

In [5]:
if "datetime" in df.columns:
    df = df.set_index("datetime")

df = df.sort_index()

missing_per_col_station = (
    df.groupby("location")
      .apply(lambda x: x.isna().mean() * 100)
      .round(1)
      .loc[:, lambda x: (x != 0).any()]
)

print(missing_per_col_station)

                     no2  pm10    so2
location                             
Gyor 2 Ifjusag      34.0   0.3    0.0
Gyor Ifjusag        35.8   0.4    0.0
Gyor Szent Istvan    0.3   0.0    6.2
VS                 100.0  59.8  100.0


## Train és teszt adatok létrehozása

In [6]:
split_date = "2025-09-01"

train = df[df.index < split_date]
test  = df[df.index >= split_date]

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

In [7]:
# --- LOG TRANSFORM TARGET ---
y_train_log = np.log1p(y_train)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print("Train period:", train.index.min(), "→", train.index.max())
print("Test period :", test.index.min(), "→", test.index.max())

Train: (25245, 34)
Test : (15505, 34)
Train period: 2024-03-02 20:00:00+00:00 → 2025-08-31 23:00:00+00:00
Test period : 2025-09-01 00:00:00+00:00 → 2026-03-01 18:00:00+00:00


## Modell paraméterezés

In [8]:
models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=500,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=7,
        max_features=None,
        random_state=42
    ),
    
    "HistGB": HistGradientBoostingRegressor(
        max_iter=323,
        learning_rate=0.0798368301055767,
        max_depth=6,
        min_samples_leaf=81,
        l2_regularization=0.4004396597514025,
        max_bins=219,
        random_state=42
    )
}

## Modell tanítás

In [9]:
print("\nTraining models...")

results = []

for name, model in models.items():
    
    model.fit(X_train, y_train_log)
    
    pred_log = model.predict(X_test)
    pred = np.maximum(0, np.expm1(pred_log))
    
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    
    results.append({
        "model": name,
        "MAE": mae,
        "R2": r2
    })

results_df = pd.DataFrame(results).sort_values("MAE")
results_df


Training models...


,model,MAE,R2
1,HistGB,3.626847,0.884249
0,RandomForest,3.727489,0.873543


## Legjobb modell kiválasztása

In [10]:

best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

print(f"\nBest model: {best_model_name}")


Best model: HistGB


## Metrikák

In [ ]:
# --- TRAIN PRED ---
train_pred_log = best_model.predict(X_train)
train_pred = np.maximum(0, np.expm1(train_pred_log))

# --- TEST PRED ---
test_pred_log  = best_model.predict(X_test)
test_pred = np.maximum(0, np.expm1(test_pred_log))

# --- METRICS ---
train_mae = mean_absolute_error(y_train, train_pred)
test_mae  = mean_absolute_error(y_test, test_pred)


baseline_pred = X_test["pm25_lag1"]
baseline_mae = mean_absolute_error(y_test, baseline_pred)


train_r2 = r2_score(y_train, train_pred)
test_r2  = r2_score(y_test, test_pred)

print("\n=== WITH VS ===")
print(f"Baseline MAE: {baseline_mae:.3f}", "µg/m³")
print(f"Train MAE: {train_mae:.3f}", "µg/m³")
print(f"Test  MAE: {test_mae:.3f}", "µg/m³")
print(f"Train R2 : {train_r2:.3f}")
print(f"Test  R2 : {test_r2:.3f}")


=== WITH VS ===
Baseline MAE: 5.012 µg/m³
Train MAE: 1.947 µg/m³
Test  MAE: 3.627 µg/m³
Train R2 : 0.888
Test  R2 : 0.884


## VS eldobása

In [12]:
df_no_vs = df[df["location"] != "VS"].copy()

train = df_no_vs[df_no_vs.index < split_date]
test  = df_no_vs[df_no_vs.index >= split_date]

X_train = train[FEATURES]
y_train = train[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

y_train_log = np.log1p(y_train)

## Modell újratanítása

In [13]:
models_no_vs = {
    "RandomForest": RandomForestRegressor(
        n_estimators=500,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=7,
        max_features=None,
        random_state=42
    ),
    
    "HistGB": HistGradientBoostingRegressor(
        max_iter=323,
        learning_rate=0.0798368301055767,
        max_depth=6,
        min_samples_leaf=81,
        l2_regularization=0.4004396597514025,
        max_bins=219,
        random_state=42
    )
}


print("\nTraining models...")
results = []

for name, model in models_no_vs.items():
    
    model.fit(X_train, y_train_log)
    
    pred_log = model.predict(X_test)
    pred = np.maximum(0, np.expm1(pred_log))
    
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    
    results.append({
        "model": name,
        "MAE": mae,
        "R2": r2
    })

results_df_no_vs = pd.DataFrame(results).sort_values("MAE")
results_df_no_vs


Training models...


,model,MAE,R2
0,RandomForest,3.831896,0.843966
1,HistGB,3.843037,0.840850


## Legjobb modell (NO VS)

In [14]:
best_model_name = results_df_no_vs.iloc[0]["model"]
best_model = models_no_vs[best_model_name]

print(f"\nBest model (NO VS): {best_model_name}")


Best model (NO VS): RandomForest


## Metrikák

In [ ]:
train_pred_log = best_model.predict(X_train)
train_pred = np.maximum(0, np.expm1(train_pred_log))

test_pred_log  = best_model.predict(X_test)
test_pred = np.maximum(0, np.expm1(test_pred_log))

train_mae = mean_absolute_error(y_train, train_pred)
test_mae  = mean_absolute_error(y_test, test_pred)

baseline_pred = X_test["pm25_lag1"]
baseline_mae = mean_absolute_error(y_test, baseline_pred)

train_r2 = r2_score(y_train, train_pred)
test_r2  = r2_score(y_test, test_pred)

print("\n=== WITHOUT VS ===")
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"Train MAE: {train_mae:.3f}")
print(f"Test  MAE: {test_mae:.3f}")
print(f"Train R2 : {train_r2:.3f}")
print(f"Test  R2 : {test_r2:.3f}")


=== WITHOUT VS ===
Baseline MAE: 5.079
Train MAE: 1.429
Test  MAE: 3.832
Train R2 : 0.854
Test  R2 : 0.844


# Eredmény
<br>
=== WITH VS ===<br>
<br>
Best model: HistGB<br>
Baseline MAE: 5.012 µg/m³<br>
Train MAE: 1.947 µg/m³<br>
Test  MAE: 3.627 µg/m³<br>
Train R2 : 0.888<br>
Test  R2 : 0.884<br>
<br>
<br>
=== WITHOUT VS ===<br>
<br>
Best model: RandomForest<br>
Baseline MAE: 5.079<br>
Train MAE: 1.429<br>
Test  MAE: 3.832<br>
Train R2 : 0.854<br>
Test  R2 : 0.844<br>
<br>


A VS állomás megtartása javította a modell teljesítményét, annak ellenére, hogy ennél az állomásnál több változó hiányzik.

Érdekesség, hogy a legjobb modell is megváltozott:
- VS-sel: HistGradientBoosting
- VS nélkül: RandomForest

Ez azt mutatja, hogy a modellválasztás függ az adateloszlástól és a rendelkezésre álló adatmennyiségtől.

Ez arra utal, hogy ebben az esetben a modell számára a nagyobb adatmennyiség fontosabb, mint a teljes feature-készlet, és a lag alapú változók elegendő információt biztosítanak.

A train és test metrikák közötti kisebb különbség azt is jelzi, hogy a modell stabilabbá vált és jobban generalizál.